In [5]:
# Imports
import sys
from pathlib import Path
import warnings

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.pategan.models import PATEGAN

warnings.filterwarnings("ignore", message="X does not have valid feature names")

# ---------- Custom PATEGAN for CAR ----------
class PATEGANCar(PATEGAN):
    def __init__(self):
        super().__init__(
            epsilon=5.0,     # relaxed privacy → better scores
            delta=1e-5,
            num_teachers=5,  # fewer teachers → more data per teacher
            niter=5000,      # you can change to 1000/8000 later
            batch_size=64,
            learning_rate=5e-4,
            lambda_gp=5.0,
            random_state=42,
        )

# ---------------- Preprocess data (CAR) ----------------
dataset_path = ROOT / "raw_data" / "car.csv"
output_path = ROOT / "discretized_data" / "car.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path))

# ---------------- Run Train/Test/Synthetic pipeline ----------------
input_csv = str(output_path)          # discretised CAR data
output_dir = str(ROOT / "sample_data" / "car")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "car" / "pategan")

model_preview = PATEGANCar()
print("PATEGAN will train for:", model_preview.niter, "iterations")

pipeline = TrainTestSplitPipeline(model=PATEGANCar)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\car.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\car.csv
PATEGAN will train for: 5000 iterations
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
Remapped y classes: [0, 1, 2, 3] -> [0, 1, 2, 3]
Loaded training data: X shape=(1382, 6), y shape=(1382,)
Training PATE-GAN with ε=5.0, δ=1e-05
Teachers: 5, Iterations: 5000


Training: 100%|██████████| 5000/5000 [00:37<00:00, 134.31it/s]


Training completed!

Generating 1382 synthetic samples...
[PATEGAN] Adding 3 dummy samples to cover classes: [0, 1, 2]
Remapped y_test.csv to match synthetic data encoding
Saved x_synth.csv to C:\Users\Prabu\Downloads\Katabatic\synthetic\car\pategan\x_synth.csv
Saved y_synth.csv to C:\Users\Prabu\Downloads\Katabatic\synthetic\car\pategan\y_synth.csv
Saved metadata.json to C:\Users\Prabu\Downloads\Katabatic\synthetic\car\pategan\metadata.json

Results saved to: Results\car\pategan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.1850
F1 Score: 0.2064

MLP:
Accuracy: 0.3179
F1 Score: 0.3407

RF:
Accuracy: 0.2659
F1 Score: 0.3515

XGBoost:
Accuracy: 0.1821
F1 Score: 0.0806
Train test split pipeline executed successfully.


C:\Users\Prabu\AppData\Local\pypoetry\Cache\virtualenvs\katabatic-AJr3_ocI-py3.11\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:55:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
